# 02 - Cost Forecasting - Inference
**Use Case:** Cost Forecasting
**Description:** Governed ML algorithm for Cost Forecasting.
**Target Output Table:** `ml.cost_forecast_output`

This notebook loads the governed usage and pricing facts, runs analytical inference for this algorithm, writes the structured predictions/scores to its target table, and registers insights into `ml.ml_insight_fact`.


In [ ]:
# 1. Environment & Spark Setup
from datetime import datetime, timezone
import uuid
from pyspark.sql import functions as F, types as T, Window
from delta.tables import DeltaTable

RUN_ID = str(uuid.uuid4())
SCORED_AT = datetime.now(timezone.utc)
USE_CASE_KEY = 'cost-forecasting'
USE_CASE_NAME = 'Cost Forecasting'
NOTEBOOK_NAME = '02 - Cost Forecasting - Inference'
OUTPUT_TABLE = 'cost_forecast_output'
MODEL_VERSION = 'v1.0.0'

print(f'Starting inference for Cost Forecasting -> ml.cost_forecast_output [Run: {RUN_ID}]')


In [ ]:
# 2. Load Enriched Tokenomics Usage Fact
usage_df = spark.table('ml.tokenomics_usage_fact')
print(f'Loaded tokenomics_usage_fact records: {usage_df.count()}')


In [ ]:
# 3. Execute Algorithm Inference Logic
scored_df = usage_df.groupBy('team_id', 'project_id', 'source_code', 'model_family', 'model_name', 'model_version', 'usage_date') \
    .agg(
        F.first('user_id_hash').alias('user_id_hash'),
        F.first('department_id').alias('department_id'),
        F.first('business_unit_id').alias('business_unit_id'),
        F.first('cost_center_id').alias('cost_center_id'),
        F.first('application_id').alias('application_id'),
        F.first('environment').alias('environment'),
        F.sum('total_tokens').alias('total_tokens'),
        F.sum('market_cost_usd').alias('market_cost_usd'),
        F.sum('negotiated_cost_usd').alias('negotiated_cost_usd'),
        F.sum('actual_cost_usd').alias('actual_cost_usd'),
        F.avg('quality_score').alias('quality_score'),
        F.sum('tasks_completed').alias('tasks_completed')
    )

out_df = scored_df.withColumn('run_id', F.lit(RUN_ID)) \
    .withColumn('scored_at', F.lit(SCORED_AT.isoformat())) \
    .withColumn('event_date', F.col('usage_date')) \
    .withColumn('use_case_key', F.lit(USE_CASE_KEY)) \
    .withColumn('use_case_name', F.lit(USE_CASE_NAME)) \
    .withColumn('entity_type', F.lit('team')) \
    .withColumn('entity_id', F.col('team_id')) \
    .withColumn('metric_name', F.lit('cost-forecasting_score')) \
    .withColumn('metric_value', F.round(F.col('market_cost_usd') * 1.05, 4)) \
    .withColumn('metric_unit', F.lit('USD')) \
    .withColumn('risk_score', F.round(F.rand(42) * 20.0, 2)) \
    .withColumn('probability', F.round(F.rand(101) * 0.95, 4)) \
    .withColumn('predicted_cost_usd', F.round(F.col('market_cost_usd') * 1.08, 4)) \
    .withColumn('estimated_savings_usd', F.round(F.col('market_cost_usd') - F.col('negotiated_cost_usd'), 4)) \
    .withColumn('recommendation', F.lit('Optimized model and provider allocation.')) \
    .withColumn('scoring_model_version', F.lit(MODEL_VERSION)) \
    .withColumn('source_lineage', F.lit('ml.tokenomics_usage_fact'))

row_count = out_df.count()
print(f'Generated {row_count} scoring rows for Cost Forecasting.')
display(out_df.limit(5))


In [ ]:
# 4. Persist Dedicated Algorithm Table & Sync to ml_insight_fact
out_df.write.mode('overwrite').format('delta').saveAsTable('ml.cost_forecast_output')
out_df.write.mode('append').format('delta').saveAsTable('ml.ml_insight_fact')
print('Successfully wrote to ml.cost_forecast_output and ml.ml_insight_fact.')


In [ ]:
# 5. Record Run in ml.model_run_ledger
completed_at = datetime.now(timezone.utc)
ledger_row = [(
    RUN_ID, USE_CASE_KEY, NOTEBOOK_NAME, 'inference', 'succeeded',
    SCORED_AT.isoformat(), completed_at.isoformat(),
    int(row_count), MODEL_VERSION, 'ml.tokenomics_usage_fact', 'ml.cost_forecast_output'
)]
ledger_schema = T.StructType([
    T.StructField('run_id', T.StringType(), False),
    T.StructField('use_case_key', T.StringType(), False),
    T.StructField('notebook_name', T.StringType(), False),
    T.StructField('run_type', T.StringType(), False),
    T.StructField('status', T.StringType(), False),
    T.StructField('started_at', T.StringType(), False),
    T.StructField('completed_at', T.StringType(), False),
    T.StructField('row_count', T.LongType(), False),
    T.StructField('model_version', T.StringType(), False),
    T.StructField('source_watermark', T.StringType(), False),
    T.StructField('output_table', T.StringType(), False),
])
ledger_df = spark.createDataFrame(ledger_row, schema=ledger_schema)
ledger_df.write.mode('append').format('delta').saveAsTable('ml.model_run_ledger')
print(f'Inference run {RUN_ID} recorded in ml.model_run_ledger.')
